In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:


import os
print("Violent videos:",
      len(os.listdir("/content/drive/MyDrive/violence detection ai/dataset/all_videos/violent")))
print("NonViolent videos:",
      len(os.listdir("/content/drive/MyDrive/violence detection ai/dataset/all_videos/nonviolent")))

Violent videos: 1115
NonViolent videos: 1000


In [3]:
import os, shutil, random

source_dir  = "/content/drive/MyDrive/violence detection ai/dataset/all_videos"
output_dir  = "/content/drive/MyDrive/violence detection ai/dataset"
classes     = ["violent", "nonviolent"]
split_ratio = (0.7, 0.15, 0.15)

for cls in classes:
    files = os.listdir(os.path.join(source_dir, cls))
    random.shuffle(files)
    total     = len(files)
    train_end = int(total * split_ratio[0])
    val_end   = int(total * (split_ratio[0] + split_ratio[1]))
    splits = {
        "train": files[:train_end],
        "val":   files[train_end:val_end],
        "test":  files[val_end:]
    }
    for split, split_files in splits.items():
        split_path = os.path.join(output_dir, split, cls)
        os.makedirs(split_path, exist_ok=True)
        for file in split_files:
            shutil.copy(
                os.path.join(source_dir, cls, file),
                os.path.join(split_path, file)
            )
print(" Dataset split done!")


for split in ["train", "val", "test"]:
    for cls in ["violent", "nonviolent"]:
        path = os.path.join(output_dir, split, cls)
        print(f"{split}/{cls}: {len(os.listdir(path))} videos")

 Dataset split done!
train/violent: 1090 videos
train/nonviolent: 969 videos
val/violent: 436 videos
val/nonviolent: 385 videos
test/violent: 438 videos
test/nonviolent: 368 videos


In [4]:
import os, cv2, numpy as np, shutil

IMG_SIZE        = 224
SEQUENCE_LENGTH = 16
CLASSES         = ["nonviolent", "violent"]
BASE_PATH       = "/content/dataset"

if not os.path.exists("/content/dataset"):
    shutil.copytree(
        "/content/drive/MyDrive/violence detection ai/dataset",
        "/content/dataset"
    )
print(" Dataset ready!")

def extract_frames_from_video(video_path):
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    if total < SEQUENCE_LENGTH:
        cap.release()
        return []
    step = total // SEQUENCE_LENGTH
    for i in range(SEQUENCE_LENGTH):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frame = frame / 255.0
        frames.append(frame)
    cap.release()
    return frames

def video_generator(split, batch_size=8):
    all_samples = []
    for label, cls in enumerate(CLASSES):
        folder = os.path.join(BASE_PATH, split, cls)
        for video_file in os.listdir(folder):
            all_samples.append((
                os.path.join(folder, video_file),
                label
            ))
    np.random.shuffle(all_samples)
    print(f"{split}: {len(all_samples)} total videos")
    while True:
        for i in range(0, len(all_samples), batch_size):
            batch = all_samples[i : i + batch_size]
            X, y = [], []
            for video_path, label in batch:
                frames = extract_frames_from_video(video_path)
                if len(frames) == SEQUENCE_LENGTH:
                    X.append(frames)
                    y.append(label)
            if X:
                yield np.array(X), np.array(y)

# Verify input shape
train_gen        = video_generator("train", batch_size=8)
X_batch, y_batch = next(train_gen)
print(f"Input Shape: {X_batch.shape}")
print(" Phase 1 Complete!")

 Dataset ready!
train: 2059 total videos
Input Shape: (8, 16, 224, 224, 3)
 Phase 1 Complete!
